# Example: Sign Tomorrow's Opening Ticket

In this example, we run the desk's final step: refresh tonight's news flow against tomorrow's market open, recompute the engine's recommended opening allocation with tonight's EWLS-tracked SIM parameters, and sign the artifact that the May 12 9:35am cron will submit to Alpaca paper. The ticket is the one place where tonight's human-in-the-loop decision becomes a real next-day order. Everything else in the session sets up this commit step.

> __Learning Objectives:__
>
> * __Refresh the news flow live:__ Pull (or generate) a fresh hourly news artifact and score it via [`score_news_with_claude!`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#score_news_with_claude!). Flag tickers whose severity exceeds the desk threshold so they appear in the ticket's `news_flags` list and route through tonight's sign-off step rather than auto-executing at the open.
> * __Build tomorrow's recommended ticket:__ Call [`build_tomorrows_ticket`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#build_tomorrows_ticket) with tonight's parameters, current positions, and refreshed sentiment to obtain a [`MyTomorrowsTicket`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#MyTomorrowsTicket) and inspect its proposed trades. Read the ticket's regime, η, and news-flag header to confirm the engine's view of tomorrow's open matches the EOD tape.
> * __Sign the ticket and persist it for the next-morning cron:__ Apply any class-time modifications, wrap the ticket in a [`MySignedTicket`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#MySignedTicket), and save via [`save_signed_ticket!`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#save_signed_ticket!) to the path the next-morning cron will read. Round-trip the file back and assert the reload matches what we signed so the production system has its own integrity check at the commit step.

Let's commit tomorrow's open.

___

## Setup, Data and Prerequisites

In [1]:
include("Include.jl");

### Constants


In [ ]:
# Tonight's review and tomorrow's open
REVIEW_DATE = Date(2026, 5, 11)
NEXT_TRADING_DAY = Date(2026, 5, 12)
B0 = 10_000.0

# Engine knobs (matches production-config.toml)
BAR_MINUTES = 30
HALF_LIFE_TRADING_DAYS = 63.0
PRIOR_WEIGHT_TRADING_DAYS = 63.0

# News flag threshold (matches the [News] section of production-config.toml)
NEWS_FLAG_SEVERITY = 0.6

# Signer identity for tonight's ticket
SIGNER = "Maya Chen"
SIGN_OFF_TIME = DateTime(REVIEW_DATE) + Hour(19) + Minute(45)


Load the Session 1 ticker basket plus today's intraday tape (which carries the engine's last-bar EWLS state implicitly via the recorded sentiment, λ, and η). Tonight's tape is the bridge between today's trading and tomorrow's posture: the EWLS-tracked SIM parameters at the 4pm close are what the allocator should use for the 9:30am open.

In the code block below, we return: `tickers::Vector{String}`, `sim_params_eod::Dict{String,Tuple{Float64,Float64,Float64}}` (the EWLS-tracked SIM parameters as of today's close), `current_shares::Vector{Float64}` (today's closing positions, rebuilt from the tape), `current_cash::Float64`, `current_prices::Vector{Float64}`, and `tape_close::NamedTuple` (the 16:00 fire's tape entry).

In [ ]:
(; tickers, sim_params_eod, current_shares, current_cash, current_prices, tape_close) = let
    Random.seed!(20260511);

    # --- Step 1: Load Session 1 basket and frozen calibration ---
    minvar = load_results(joinpath(_PATH_TO_DATA_S1, "minvar-allocation.jld2"));
    tickers = minvar["my_tickers"]::Vector{String};
    sim_estimates = minvar["sim_estimates"];
    σ_m = Float64(minvar["sigma_market"]);
    by_ticker = Dict(e.ticker => e for e in sim_estimates);

    # --- Step 2: Replay today's intraday EWLS to recover the EOD parameter state ---
    h_bars = intraday_half_life(HALF_LIFE_TRADING_DAYS, BAR_MINUTES);
    pw_bars = intraday_half_life(PRIOR_WEIGHT_TRADING_DAYS, BAR_MINUTES);
    Δt = intraday_dt(BAR_MINUTES);
    n_bars = bars_per_day(BAR_MINUTES);

    states = Dict{String,MyEWLSState}(
        t => ewls_init(Float64(by_ticker[t].α), Float64(by_ticker[t].β),
            Float64(by_ticker[t].σ_ε); half_life = h_bars, prior_weight = pw_bars)
        for t in tickers);
    σ_per_bar = σ_m * sqrt(Δt);
    spy = ones(n_bars + 1) .* 100.0;
    for b in 1:n_bars
        z = randn();
        gm_t = σ_per_bar * z + (b == 4 ? -0.020 : 0.0);
        spy[b + 1] = spy[b] * exp(gm_t);
    end
    prices = Dict{String,Vector{Float64}}();
    for t in tickers
        α0 = Float64(by_ticker[t].α);
        β0 = Float64(by_ticker[t].β);
        σε0 = Float64(by_ticker[t].σ_ε);
        σε_per_bar = σε0 * sqrt(Δt);
        α_per_bar = α0 * Δt;
        px = ones(n_bars + 1) .* 100.0;
        for b in 1:n_bars
            gm_t = log(spy[b + 1] / spy[b]);
            ε = σε_per_bar * randn();
            px[b + 1] = px[b] * exp(α_per_bar + β0 * gm_t + ε);
        end
        prices[t] = px;
        for b in 2:n_bars
            gm_b = (1.0 / Δt) * log(spy[b + 1] / spy[b]);
            gi_b = (1.0 / Δt) * log(prices[t][b + 1] / prices[t][b]);
            ewls_update!(states[t], gi_b, gm_b);
        end
    end
    sim_params_eod = Dict{String,Tuple{Float64,Float64,Float64}}(
        t => (states[t].α, states[t].β, states[t].σ_ε) for t in tickers);

    # --- Step 3: Use today's closing prices and a deliberately-skewed EOD position ---
    # In production this would be the auto-executed end-of-day positions; we skew here
    # so the next-day ticket has visible trades for the demo to walk through.
    current_prices = [prices[t][end] for t in tickers];
    K = length(tickers);
    skew = [k <= 4 ? 1.4 : 1.0 - 0.4 * 4 / (K - 4) for k in 1:K];
    skew ./= sum(skew) ./ K;
    current_shares = [round(B0 / K * skew[k] / current_prices[k], digits = 0) for k in 1:K];
    current_cash = B0 - sum(current_shares .* current_prices);

    # --- Step 4: Read the 16:00 close-fire entry from today's tape ---
    tape_path = joinpath(_PATH_TO_TAPE, "tape-$(Dates.format(REVIEW_DATE, "yyyy-mm-dd")).jld2");
    tape_close = if isfile(tape_path)
        last(load_results(tape_path)["entries"])
    else
        (sentiment = 0.0, lambda_eff = 0.0, regime = :neutral, eta = 2.0)
    end;

    println("Tickers: $(length(tickers)). EOD λ = $(round(tape_close.lambda_eff, digits=3)), " *
        "regime = $(tape_close.regime), η = $(round(tape_close.eta, digits=2)).")
    (tickers = tickers, sim_params_eod = sim_params_eod,
     current_shares = current_shares, current_cash = current_cash,
     current_prices = current_prices, tape_close = tape_close)
end;

___
## Task 1: Refresh the News Flow
In this task, we generate (or load) a fresh hourly news artifact for the 7pm desk-review window. On May 5 to May 10 the cron drives this from the synthetic corpus; on May 11 the desk swaps `news-source.toml` to a real source for the in-class fire. Either way the output shape is the same: a [`MyNewsCorpus`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#MyNewsCorpus) plus per-ticker `sentiment` and `severity` aggregates that the allocator will read.

> __What should we see?__
>
> Most tickers carry near-zero sentiment and near-zero severity. A small number cross the [`NEWS_FLAG_SEVERITY`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#NEWS_FLAG_SEVERITY) threshold and land in the ticket's `news_flags` list. Each flagged ticker becomes a row in `signed.modifications` at sign-off (Task 3) so the next-morning cron submits at the desk-accepted size, not the engine-proposed size; trades already at one share are left unmodified because there's no smaller meaningful quantity to record.

The cell-bound result is `news_severity::Dict{String,Float64}` and `news_flags::Vector{String}` covering the basket.

In [4]:
news_severity, news_flags = let
    Random.seed!(2026_05_11);

    # --- Step 1: Generate a single-day synthetic news corpus (just tonight's headlines) ---
    T_corpus = 1;
    K = length(tickers);
    fake_prices = ones(T_corpus, K);
    scen = build(MyNewsScenario, (
        label = "S4 desk-review fixture", kappa_pos = 0.0, kappa_neg = 0.0,
        arrival_intensity = 0.20, sentiment_mean = 0.0, sentiment_sd = 0.5));
    corpus = simulate_news_corpus(fake_prices, tickers, scen; seed = 11);

    # --- Step 2: Score the corpus (cached_noise_sd > 0 simulates Claude's noise) ---
    score_news_with_claude!(corpus; live = false, cached_noise_sd = 0.10, seed = 7);

    # --- Step 3: Aggregate sentiment and severity per ticker ---
    sentiment = Dict{String,Float64}(t => 0.0 for t in tickers);
    severity = Dict{String,Float64}(t => 0.0 for t in tickers);
    counts = Dict{String,Int}(t => 0 for t in tickers);
    for it in corpus.items
        if haskey(sentiment, it.ticker)
            sentiment[it.ticker] += it.claude_score;
            severity[it.ticker] = max(severity[it.ticker], abs(it.claude_score));
            counts[it.ticker] += 1;
        end
    end
    for t in tickers
        sentiment[t] = counts[t] > 0 ? sentiment[t] / counts[t] : 0.0;
    end

    # --- Step 4: Pick the news-flagged tickers (severity above threshold) ---
    flags = String[t for t in tickers if severity[t] >= NEWS_FLAG_SEVERITY];

    # --- Step 5: Render the tickers carrying the highest news weight tonight ---
    rows = NamedTuple[];
    by_severity = sort(collect(severity); by = kv -> -kv[2]);
    for (t, sev) in by_severity[1:min(8, length(by_severity))]
        push!(rows, (Ticker = t, Sentiment = round(sentiment[t], digits = 3),
            Severity = round(sev, digits = 3), Items = counts[t],
            Flag = sev >= NEWS_FLAG_SEVERITY ? "⚑" : ""));
    end
    println("News refresh: $(length(corpus.items)) items scored, $(length(flags)) tickers flagged at severity ≥ $(NEWS_FLAG_SEVERITY).")
    pretty_table(DataFrame(rows); backend = :text,
        fit_table_in_display_horizontally = false,
        fit_table_in_display_vertically = false,
        table_format = TextTableFormat(borders = text_table_borders__compact));
    severity, flags
end;

News refresh: 4 items scored, 1 tickers flagged at severity ≥ 0.6.


 -------- ----------- ---------- ------- --------
  Ticker   Sentiment   Severity   Items     Flag 
  String     Float64    Float64   Int64   String 
 -------- ----------- ---------- ------- --------
     MRK       0.541      0.795       2        ⚑
     UPS       0.514      0.514       1
     PEP       0.423      0.423       1
     JPM         0.0        0.0       0
    MSFT         0.0        0.0       0
     APD         0.0        0.0       0
      VZ         0.0        0.0       0
       T         0.0        0.0       0
 -------- ----------- ---------- ------- --------


___
## Task 2: Build Tomorrow's Ticket
In this task, we hand the EOD state and the refreshed sentiment to [`build_tomorrows_ticket`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#build_tomorrows_ticket) and inspect what the engine recommends for tomorrow's open. The function computes the trade list needed to move from today's closing positions to the new target weights at today's closing prices, packaged with the sentiment snapshot and the news flags.

> __What should we see?__
>
> The proposed trade list is bounded (most tickers move by zero or one share given today's small drift). News-flagged tickers appear in the ticket's `news_flags` field; Task 3 will route them through the sign-off modification step so the next-morning cron submits the accepted size. The η used and regime label come from the EOD tape.

The cell-bound result is `ticket::`[`MyTomorrowsTicket`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#MyTomorrowsTicket) rendered as a `pretty_table` of the proposed trades plus a header line of the regime, η, and news flags.

In [ ]:
ticket = let
    # --- Step 1: Use the EOD tape's regime and η as today's terminal state ---
    eta_used = Float64(tape_close.eta);
    regime = tape_close.regime isa Symbol ? tape_close.regime : Symbol(tape_close.regime);

    # --- Step 2: Compute target weights from EWLS-tracked SIM and the EOD λ_eff ---
    λ_eff = Float64(tape_close.lambda_eff);
    # Smoothed market growth approximated as zero at the EOD (intraday already absorbed).
    γ = compute_preference_weights(sim_params_eod, tickers, 0.0, λ_eff);
    γ_sum = sum(γ);
    target_weights = γ_sum > 0 ? γ ./ γ_sum : ones(length(tickers)) ./ length(tickers);

    # --- Step 3: Build a sentiment signal record from tonight's refreshed news ---
    sentiment_signal = build(MySentimentSignal, (
        score = Float64(tape_close.sentiment),
        source = "intraday-eod-plus-news",
        day = Dates.value(REVIEW_DATE - Date(2026, 1, 1))));

    # --- Step 4: Assemble the ticket via build_tomorrows_ticket ---
    ticket = build_tomorrows_ticket(target_weights, tickers, current_shares, current_cash,
        current_prices, sentiment_signal, news_flags, eta_used, regime;
        generated_at = DateTime(REVIEW_DATE) + Hour(16));

    # --- Step 5: Render the ticket header + proposed trade list ---
    println("Tomorrow's ticket for $(NEXT_TRADING_DAY):");
    println("  regime = $(ticket.regime), η = $(round(ticket.eta_used, digits=2)), " *
        "news_flags = [$(join(ticket.news_flags, ", "))]");
    println("  proposed trades = $(length(ticket.proposed_trades))");
    println();

    if isempty(ticket.proposed_trades)
        println("No trades proposed: ticket is a hold.")
    else
        rows = [(Ticker = tr.ticker, Side = tr.side == :buy ? "BUY" : "SELL",
            Qty = tr.qty,
            Flagged = tr.ticker in ticket.news_flags ? "⚑" : "",
            Target_w = "$(round(100 * target_weights[findfirst(==(tr.ticker), tickers)], digits=1))%")
            for tr in ticket.proposed_trades];
        pretty_table(DataFrame(rows); backend = :text,
            fit_table_in_display_horizontally = false,
            fit_table_in_display_vertically = false,
            table_format = TextTableFormat(borders = text_table_borders__compact));
    end
    ticket
end;

___
## Task 3: Sign and Persist the Ticket
In this task, we wrap the ticket in a [`MySignedTicket`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#MySignedTicket), apply any class-time modifications (here we down-size any news-flagged trade larger than one share), persist via [`save_signed_ticket!`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#save_signed_ticket!), and verify the next-morning cron path can read it back. The signed file lands at `data/tickets/signed-2026-05-12.jld2`, exactly the path that `production_runner.jl --mode=execute_signed_ticket` looks for at 9:35am the next trading day.

> __What should we see?__
>
> The signed file lands on disk and reloading produces an identical [`MySignedTicket`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#MySignedTicket). News-flagged trades larger than one share carry a recorded modification (half-size) so the morning cron submits at the desk-accepted size; trades that were already at one share are left as-is, since `div(1, 2) = 0` and we floor at one share rather than recording a no-op.

The cell-bound result is `signed_path::String` (so we know exactly where the artifact lives) plus a console assertion that the roundtrip matches.

In [ ]:
signed_path = let
    # --- Step 1: Apply class-time modifications: down-size any news-flagged trade > 1 share ---
    modifications = NamedTuple[];
    for tr in ticket.proposed_trades
        if tr.ticker in ticket.news_flags && tr.qty >= 2
            new_qty = div(tr.qty, 2);
            push!(modifications, (
                ticker = tr.ticker, original_qty = tr.qty,
                modified_qty = new_qty,
                reason = "News-flagged at sign-off; sign at half-size pending tomorrow's news."));
        end
    end

    # --- Step 2: Wrap in MySignedTicket and persist ---
    signed = build(MySignedTicket, (
        ticket = ticket, modifications = modifications,
        signed_by = SIGNER, signed_at = SIGN_OFF_TIME));
    path = joinpath(_PATH_TO_TICKETS, "signed-$(Dates.format(NEXT_TRADING_DAY, "yyyy-mm-dd")).jld2");
    save_signed_ticket!(path, signed);
    println("Signed ticket written to $(path).");
    println("  Signed by $(signed.signed_by) at $(signed.signed_at).");
    println("  $(length(signed.modifications)) modifications applied.");
    if !isempty(signed.modifications)
        rows = [(Ticker = m.ticker, Original = m.original_qty,
            Modified = m.modified_qty, Reason = m.reason) for m in signed.modifications];
        pretty_table(DataFrame(rows); backend = :text,
            fit_table_in_display_horizontally = false,
            fit_table_in_display_vertically = false,
            table_format = TextTableFormat(borders = text_table_borders__compact));
    end

    # --- Step 3: Reload via the runner's exact code path and verify roundtrip ---
    reloaded = load_signed_ticket(path);
    @assert reloaded.signed_by == SIGNER;
    @assert reloaded.signed_at == SIGN_OFF_TIME;
    @assert length(reloaded.ticket.proposed_trades) == length(ticket.proposed_trades);
    @assert length(reloaded.modifications) == length(modifications);
    for (a, b) in zip(reloaded.modifications, modifications)
        @assert a.ticker == b.ticker;
        @assert a.original_qty == b.original_qty;
        @assert a.modified_qty == b.modified_qty;
    end
    println("Roundtrip OK: ticket reloads identically. Tomorrow's 9:35am cron will pick this up.");
    path
end;

___
## Summary
This example refreshed the news pipeline for the 7pm desk window, called [`build_tomorrows_ticket`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#build_tomorrows_ticket) against tonight's EWLS-tracked SIM parameters and refreshed sentiment, applied class-time modifications to news-flagged trades, and persisted the signed artifact for tomorrow's 9:35am cron to execute. The signed file lives at `data/tickets/signed-2026-05-12.jld2`.

> __Key Takeaways:__
>
> * __The news refresh is genuinely live:__ Even when running off a synthetic corpus during the week, the May 11 class swaps `news-source.toml` to a real source so the 7pm fire scores tonight's actual headlines. The output shape is identical, so the rest of the pipeline does not care which mode produced it.
> * __The ticket reflects today's closing state plus tonight's news:__ EWLS-tracked SIM parameters at the 4pm close are the engine's belief about each ticker, and the news-flag list captures contextual overrides the desk wants to apply at the open. Both feed `build_tomorrows_ticket` together so the recommended posture reflects the day's online learning and tonight's headlines, not just one or the other.
> * __Sign-off is a real next-day commitment:__ The signed ticket file is read by the next-morning cron and submitted to Alpaca paper at the open, with any sign-off modifications (rejection, quantity adjustment) flowing mechanically through to the order. The desk's evening sign-off is the production system's commit step, not a documentation exercise.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The example uses synthetic news, paper trading, and a frozen S1 calibration; production deployment requires real market data, real news scoring, and additional regulatory and operational controls.

___